Exercise 4 — Outdoor Event Plan to Weather-Adjusted Recommendation Pipeline
Build a 3-step pipeline. Step 1 extracts the event details (event type, planned date, location, indoor/outdoor preference) from a free-form event description. Step 2 calls a weather API for the extracted location and date, and normalizes the response into structured weather data. Step 3 generates a recommendation message for the event organizer (proceed as planned / reschedule / move indoors) that combines both the event details and the weather data.

Input: event_description_text

Output: the structured event details from Step 1, the structured weather data from Step 2, and the recommendation message from Step 3

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: event_type, event_date, location, indoor_outdoor_preference)

Step 2 — Template: PromptTemplate (used to format the weather API call/query from Step 1's fields; the API response itself is normalized into structured output, not freely generated by the LLM)  |  Parser: PydanticOutputParser (fields: temperature, precipitation_chance, conditions)

Step 3 — Template: ChatPromptTemplate (system persona: "event planning assistant")  |  Parser: StrOutputParser

In [1]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class EventDetails(BaseModel):
    event_type: str = Field(
        description="Type of event being planned"
    )
    event_date: str = Field(
        description="Event date in YYYY-MM-DD format"
    )
    location: str = Field(
        description="City or location where the event takes place"
    )
    indoor_outdoor_preference: str = Field(
        description="Whether the organizer prefers indoor, outdoor, or either"
    )

In [2]:
ex4_event_parser = PydanticOutputParser(
    pydantic_object=EventDetails
)

In [3]:
ex4_event_prompt = PromptTemplate(
    template="""
You extract event details from a free-form description.

Extract:
- event_type
- event_date
- location
- indoor_outdoor_preference

Rules:
- Use only information supported by the description.
- Return event_date in YYYY-MM-DD format.
- If a required detail is missing, write "Not specified".
- Do not invent a date or location.

Event description:
{event_description_text}

{format_instructions}
""",
    input_variables=["event_description_text"],
    partial_variables={
        "format_instructions":
            ex4_event_parser.get_format_instructions()
    }
)

In [5]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [6]:
ex4_step1_chain = (
    ex4_event_prompt
    | llm
    | ex4_event_parser
)

In [7]:
event_description_text = """
We are organizing an outdoor birthday party in Pune,
India, on October 25, 2026. We prefer to keep the event
outdoors if the weather permits.
"""

event_details = ex4_step1_chain.invoke({
    "event_description_text": event_description_text
})

print(event_details.model_dump_json(indent=2))

{
  "event_type": "birthday party",
  "event_date": "2026-10-25",
  "location": "Pune, India",
  "indoor_outdoor_preference": "outdoor"
}


## Step 2 — Template: PromptTemplate (used to format the weather API call/query from Step 1's fields; the API response itself is normalized into structured output, not freely generated by the LLM)  |  Parser: PydanticOutputParser (fields: temperature, precipitation_chance, conditions)



In [8]:
import json
import requests

from datetime import date
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [9]:
class WeatherData(BaseModel):
    temperature: float = Field(
        description="Maximum daily temperature in degrees Celsius"
    )
    precipitation_chance: float | None = Field(
        description="Maximum daily precipitation probability in percent, or null if unavailable"
    )
    conditions: str = Field(
        description="Human-readable daily weather conditions"
    )

In [10]:
ex4_weather_parser = PydanticOutputParser(
    pydantic_object=WeatherData
)

In [11]:
ex4_weather_query_prompt = PromptTemplate(
    template=(
        "Retrieve the daily weather forecast for "
        "{location} on {event_date}."
    ),
    input_variables=["location", "event_date"]
)

In [12]:
weather_query = ex4_weather_query_prompt.format(
    location=event_details.location,
    event_date=event_details.event_date
)

print(weather_query)

Retrieve the daily weather forecast for Pune, India on 2026-10-25.


In [16]:
def get_weather_data(event_details: EventDetails) -> WeatherData:

    # Step A: Validate the requested date.
    try:
        event_day = date.fromisoformat(
            event_details.event_date
        )
    except ValueError as exc:
        raise ValueError(
            "Invalid event date. Use YYYY-MM-DD."
        ) from exc

    if event_details.location.strip().lower() in {
        "", "not specified", "unknown"
    }:
        raise ValueError(
            "A valid event location is required."
        )

    # Step B: Geocode the event location.
    geocoding_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={
            "name": event_details.location,
            "count": 1,
            "language": "en",
            "format": "json"
        },
        timeout=20
    )

    geocoding_response.raise_for_status()
    geocoding_data = geocoding_response.json()

    locations = geocoding_data.get("results", [])

    if not locations:
        raise ValueError(
            f"Location not found: {event_details.location}"
        )

    location = locations[0]
    latitude = location["latitude"]
    longitude = location["longitude"]

    # Step C: Request the forecast for the exact date.
    # IMPORTANT: Do not include forecast_days here.
    forecast_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": latitude,
            "longitude": longitude,
            "daily": (
                "temperature_2m_max,"
                "precipitation_probability_max,"
                "weather_code"
            ),
            "start_date": event_day.isoformat(),
            "end_date": event_day.isoformat(),
            "timezone": "auto"
        },
        timeout=20
    )

    # Show the API's actual error message if it fails.
    if not forecast_response.ok:
        raise RuntimeError(
            f"Weather API error "
            f"({forecast_response.status_code}): "
            f"{forecast_response.text}"
        )

    forecast_data = forecast_response.json()
    daily = forecast_data.get("daily")

    if not daily or not daily.get("time"):
        raise ValueError(
            "No daily forecast was returned for this date."
        )

    # Step D: Locate the requested date.
    try:
        index = daily["time"].index(event_day.isoformat())
    except ValueError as exc:
        raise ValueError(
            "The API did not return the requested date. "
            "It may be outside the available forecast range."
        ) from exc

    temperature = daily["temperature_2m_max"][index]
    precipitation = daily[
        "precipitation_probability_max"
    ][index]
    weather_code = daily["weather_code"][index]

    if temperature is None or weather_code is None:
        raise ValueError(
            "The API returned incomplete weather data."
        )

    # Step E: Convert weather codes into readable conditions.
    weather_descriptions = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        56: "Freezing drizzle",
        57: "Dense freezing drizzle",
        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",
        66: "Light freezing rain",
        67: "Heavy freezing rain",
        71: "Slight snowfall",
        73: "Moderate snowfall",
        75: "Heavy snowfall",
        77: "Snow grains",
        80: "Slight rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",
        85: "Slight snow showers",
        86: "Heavy snow showers",
        95: "Thunderstorm",
        96: "Thunderstorm with slight hail",
        99: "Thunderstorm with heavy hail"
    }

    conditions = weather_descriptions.get(
        int(weather_code),
        f"Unmapped weather code: {weather_code}"
    )

    # Step F: Normalize the API response into our schema.
    normalized_weather = {
        "temperature": temperature,
        "precipitation_chance": precipitation,
        "conditions": conditions
    }

    return ex4_weather_parser.parse(
        json.dumps(normalized_weather)
    )

In [17]:
weather_data = get_weather_data(event_details)

print(weather_data.model_dump_json(indent=2))

{
  "temperature": 32.9,
  "precipitation_chance": 8.0,
  "conditions": "Overcast"
}


In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ex4_recommendation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an event planning assistant.

Recommend one of these actions:
1. Proceed as planned.
2. Move the event indoors.
3. Reschedule the event.

Base your recommendation on the supplied event details
and actual weather data.

Guidelines:
- Use the reported temperature, precipitation probability,
  and weather conditions.
- If the event is outdoors and the precipitation chance
  is 50% or higher, recommend moving indoors if possible.
- If outdoor conditions appear risky and an indoor venue
  is unavailable, consider recommending rescheduling.
- If the precipitation chance is between 25% and 49%,
  explain the uncertainty and suggest a backup plan.
- If the weather appears suitable, proceeding may be
  reasonable, but do not guarantee perfect conditions.
- Consider the organizer's indoor/outdoor preference.
- Do not invent weather information.
- If weather data is missing, explain the limitation.
- Explain the reason for your recommendation and provide
  a practical next step.
- Make clear that daily forecast values may not represent
  conditions at the exact event time.
- Keep the message concise and professional.
"""
    ),
    (
        "user",
        """
Event details:
{event_details}

Weather data retrieved from the API:
{weather_data}

Write the recommendation for the event organizer.
"""
    )
])

In [19]:
ex4_step3_chain = (
    ex4_recommendation_prompt
    | llm
    | StrOutputParser()
)

In [20]:
recommendation = ex4_step3_chain.invoke({
    "event_details": event_details.model_dump_json(
        indent=2
    ),
    "weather_data": weather_data.model_dump_json(
        indent=2
    )
})

print(recommendation)

**Recommendation:** Proceed as planned.

**Reasoning:**  
- The forecast for 25 Oct 2026 in Pune shows a low precipitation chance of 8 % and a moderate temperature of 32.9 °C.  
- Since the precipitation probability is well below the 25 % threshold, the risk of rain is minimal.  
- The event is preferred outdoors, and the current conditions (overcast but dry) are suitable for an outdoor birthday party.

**Practical next step:**  
- Confirm the outdoor venue and ensure basic shelter options (e.g., a lightweight canopy or umbrellas) are available in case of unexpected showers.  
- Keep an eye on the daily forecast as the event date approaches, as weather can change.  

*Note: Daily forecasts may not reflect the exact conditions at the event time, so a quick check a day before is advisable.*


In [21]:
def process_event_plan(event_description_text: str):
    # Step 1: Extract event details.
    event_details = ex4_step1_chain.invoke({
        "event_description_text": event_description_text
    })

    # Validate the date before calling the weather API.
    try:
        date.fromisoformat(event_details.event_date)
    except ValueError as exc:
        raise ValueError(
            "The event description must contain a valid date "
            "in YYYY-MM-DD format."
        ) from exc

    # Format the weather query using Step 1's output.
    weather_query = ex4_weather_query_prompt.format(
        location=event_details.location,
        event_date=event_details.event_date
    )

    print("WEATHER QUERY:")
    print(weather_query)

    # Step 2: Fetch and normalize actual weather data.
    weather_data = get_weather_data(event_details)

    # Step 3: Generate the organizer's recommendation.
    recommendation = ex4_step3_chain.invoke({
        "event_details": event_details.model_dump_json(
            indent=2
        ),
        "weather_data": weather_data.model_dump_json(
            indent=2
        )
    })

    return {
        "event_details": event_details,
        "weather_data": weather_data,
        "recommendation": recommendation
    }

In [22]:
event_description_text = """
We are organizing an outdoor birthday party in Pune,
India, on October 25, 2026. We prefer to keep the event
outdoors if the weather permits.
"""

result = process_event_plan(event_description_text)

print("\nSTEP 1: EVENT DETAILS")
print(
    result["event_details"].model_dump_json(indent=2)
)

print("\nSTEP 2: WEATHER DATA")
print(
    result["weather_data"].model_dump_json(indent=2)
)

print("\nSTEP 3: FINAL RECOMMENDATION")
print(result["recommendation"])

WEATHER QUERY:
Retrieve the daily weather forecast for Pune, India on 2026-10-25.

STEP 1: EVENT DETAILS
{
  "event_type": "birthday party",
  "event_date": "2026-10-25",
  "location": "Pune, India",
  "indoor_outdoor_preference": "outdoor"
}

STEP 2: WEATHER DATA
{
  "temperature": 32.9,
  "precipitation_chance": 8.0,
  "conditions": "Overcast"
}

STEP 3: FINAL RECOMMENDATION
**Recommendation:** Proceed as planned.

**Why:**  
- The forecast shows only an 8 % chance of precipitation, well below the 25 % threshold that would trigger a move‑indoor recommendation.  
- Temperature (≈33 °C) and overcast skies suggest comfortable, non‑risky conditions for an outdoor birthday party.  

**Practical next step:**  
- Confirm all outdoor arrangements (tables, seating, décor).  
- Keep a lightweight umbrella or rain poncho on hand just in case the weather changes unexpectedly.  
- Remind guests to dress appropriately for a warm, overcast day.  

*Note:* Daily forecasts may not capture micro‑clima